In [6]:
# --- Cell 1: Paths ---
from pathlib import Path

VOC2010_ROOT = Path("/kaggle/input/datasets/xuantuan/cityscapes/VOC2010")  # TODO: adjust

MAT_DIR = Path("/kaggle/input/datasets/xuantuan/cityscapes/trainval/trainval")
LABELS_459 = Path("/kaggle/input/datasets/xuantuan/cityscapes/trainval/labels.txt")
LABELS_59 = Path("/kaggle/input/datasets/xuantuan/cityscapes/59_contexts/59_labels.txt")

OUT_MASK_DIR = Path("/kaggle/working/SegmentationClassContext")

for p in (MAT_DIR, LABELS_459, LABELS_59):
    assert p.exists(), f"MISSING: {p}"

n_mats = len(list(MAT_DIR.glob("*.mat")))
print(f"MAT_DIR     : {MAT_DIR}  ({n_mats} .mat files)")
print(f"LABELS_459  : {LABELS_459}")
print(f"LABELS_59   : {LABELS_59}")
print(f"OUT_MASK_DIR: {OUT_MASK_DIR}")


MAT_DIR     : /kaggle/input/datasets/xuantuan/cityscapes/trainval/trainval  (10103 .mat files)
LABELS_459  : /kaggle/input/datasets/xuantuan/cityscapes/trainval/labels.txt
LABELS_59   : /kaggle/input/datasets/xuantuan/cityscapes/59_contexts/59_labels.txt
OUT_MASK_DIR: /kaggle/working/SegmentationClassContext


In [7]:
# --- Cell 2: Imports ---
import numpy as np
import scipy.io
import tqdm
from PIL import Image

print("✓ Imports OK")


✓ Imports OK


In [8]:
# --- Cell 3: Build the 459 -> 60 label lookup table ---
# Verbatim logic from the verified local script: 0 = background (catch-all for anything not one
# of the 59 target classes), 1..59 = the 59 classes, sorted alphabetically to match
# mmsegmentation's PascalContextDataset59.CLASSES order (same order sam3_baseline_pc59_nollm.ipynb
# hardcodes as PC59_CLASSES).

def parse_labels_459(path):
    """labels.txt format: '<id>: <name>' per line."""
    out = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or ":" not in line:
                continue
            idx, name = line.split(":", 1)
            try:
                out[name.strip()] = int(idx.strip())
            except ValueError:
                continue
    return out


def parse_labels_59(path, dict_459):
    """59_labels.txt format: lines like '<idx>: <name>' or just '<name>'."""
    names = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if ":" in line:
                line = line.split(":", 1)[-1].strip()
            names.append(line)
    return [(name, dict_459[name]) for name in names if name in dict_459]


dict_459 = parse_labels_459(LABELS_459)
pc59_pairs = parse_labels_59(LABELS_59, dict_459)

file_names = set()
with open(LABELS_59, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line and ":" in line:
            file_names.add(line.split(":", 1)[1].strip())
mapped = {n for n, _ in pc59_pairs}
missing = file_names - mapped
assert len(pc59_pairs) == 59, f"Only {len(pc59_pairs)}/59 mapped; missing: {missing}"

pc59_pairs.sort(key=lambda x: x[0])  # alphabetical, matches PC59_CLASSES in sam3_baseline_pc59_nollm.ipynb

lut = np.zeros(max(dict_459.values()) + 1, dtype=np.uint8)  # default 0 = background
for new_idx, (name, old_idx) in enumerate(pc59_pairs, start=1):
    lut[old_idx] = new_idx

print(f"✓ LUT built. First 5 classes (pixel value 1-5): {[n for n, _ in pc59_pairs[:5]]}")
print(f"           Last 5 classes  (pixel value 55-59): {[n for n, _ in pc59_pairs[-5:]]}")


✓ LUT built. First 5 classes (pixel value 1-5): ['aeroplane', 'bag', 'bed', 'bedclothes', 'bench']
           Last 5 classes  (pixel value 55-59): ['tvmonitor', 'wall', 'water', 'window', 'wood']


In [9]:
# --- Cell 4: Convert all .mat -> PNG ---
OUT_MASK_DIR.mkdir(parents=True, exist_ok=True)

all_ids = sorted(p.stem for p in MAT_DIR.glob("*.mat"))
print(f"Converting {len(all_ids)} .mat files -> {OUT_MASK_DIR}")

skipped = 0
for img_id in tqdm.tqdm(all_ids):
    mat_path = MAT_DIR / f"{img_id}.mat"
    out_path = OUT_MASK_DIR / f"{img_id}.png"
    try:
        mat = scipy.io.loadmat(str(mat_path))
        mask = mat["LabelMap"]  # shape (H, W), int values into the 459-class raw label space
    except Exception as e:
        print(f"\nSkipped {img_id}: {e}")
        skipped += 1
        continue
    mask = mask.astype(np.int64)
    mask = np.clip(mask, 0, len(lut) - 1)  # some raw pixels can exceed the 459-class range
    out = lut[mask]
    Image.fromarray(out, mode="L").save(out_path)

print(f"\nDone. Converted: {len(all_ids) - skipped}   Skipped: {skipped}")


Converting 10103 .mat files -> /kaggle/working/SegmentationClassContext


  0%|          | 0/10103 [00:00<?, ?it/s]/tmp/ipykernel_57/1605775238.py:21: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(out, mode="L").save(out_path)
100%|██████████| 10103/10103 [01:36<00:00, 104.79it/s]


Done. Converted: 10103   Skipped: 0


In [10]:
# --- Cell 5: Verify output ---
out_pngs = list(OUT_MASK_DIR.glob("*.png"))
print(f"PNG masks written: {len(out_pngs)}")

_sample_path = out_pngs[0] if out_pngs else None
if _sample_path is not None:
    _sample = np.array(Image.open(_sample_path))
    _uniques = np.unique(_sample)
    print(f"\nSample '{_sample_path.name}' unique pixel values ({len(_uniques)}): {_uniques}")
    _unexpected = sorted(set(_uniques.tolist()) - set(range(60)))
    if _unexpected:
        print(f"[warn] unexpected values (outside [0,59]): {_unexpected}")
    else:
        print("\u2713 all values within expected range (0=background, 1-59=class)")

print("\nNext step: Save Version -> Output tab -> \"Create Dataset\" from /kaggle/working,")
print("then point PC59_GT_ROOT in sam3_baseline_pc59_nollm.ipynb at the new dataset's")
print("SegmentationClassContext/ folder. pascal_context_train.txt/val.txt are used as-is,")
print("unchanged, alongside it -- no further processing needed for the split.")


PNG masks written: 10103

Sample '2008_001871.png' unique pixel values (4): [ 0 18 23 37]
✓ all values within expected range (0=background, 1-59=class)

Next step: Save Version -> Output tab -> "Create Dataset" from /kaggle/working,
then point PC59_GT_ROOT in sam3_baseline_pc59_nollm.ipynb at the new dataset's
SegmentationClassContext/ folder. pascal_context_train.txt/val.txt are used as-is,
unchanged, alongside it -- no further processing needed for the split.


In [11]:
# --- Cell 6: Zip output for direct download ---
# Kaggle's file browser can be slow/unreliable for 10K+ individual files -- zip into one file
# so it shows up as a single downloadable item in the notebook's Output panel (no need to go
# through "Create Dataset" if you just want the file on your own machine).
import shutil

zip_path = shutil.make_archive("/kaggle/working/SegmentationClassContext", "zip", OUT_MASK_DIR)
print(f"\u2713 Zipped to {zip_path}")
print(f"Size: {Path(zip_path).stat().st_size / (1024**2):.1f} MB")
print("\nDownload it from this notebook's Output panel (or still use Save Version -> Create Dataset")
print("if you'd rather attach it directly as input to sam3_baseline_pc59_nollm.ipynb without")
print("re-uploading).")


✓ Zipped to /kaggle/working/SegmentationClassContext.zip
Size: 28.8 MB

Download it from this notebook's Output panel (or still use Save Version -> Create Dataset
if you'd rather attach it directly as input to sam3_baseline_pc59_nollm.ipynb without
re-uploading).
